# VietTTS: Vietnamese Text-to-Speech with Voice Cloning

## What is VietTTS?

[VietTTS](https://github.com/dangvansam/viet-tts) is an open-source Vietnamese text-to-speech toolkit that supports natural voice synthesis and voice cloning. It provides an **OpenAI-compatible API** server, making it a drop-in replacement for OpenAI TTS in Vietnamese applications.

### Key Features
- High-quality Vietnamese speech synthesis
- **Voice cloning** from a short reference audio clip (3-10s)
- **OpenAI API-compatible** server (`/v1/audio/speech`)
- CLI tool for quick synthesis
- Built on the **F5-TTS** framework with HiFi-GAN vocoder

### Links
- **PyPI**: `pip install viet-tts`
- **GitHub**: [dangvansam/viet-tts](https://github.com/dangvansam/viet-tts)
- **HuggingFace**: [dangvansam/viet-tts](https://huggingface.co/dangvansam/viet-tts)

### License
- Source code: Apache 2.0
- Pre-trained models / voice samples: CC BY-NC

---
**Kaggle Setup**: GPU T4 | Internet enabled

## Step 1: Install VietTTS

VietTTS is available on PyPI. The package will download pretrained models automatically on first use.

In [ ]:
!pip install -q viet-tts

## Step 2: Explore Available Voices

VietTTS ships with several built-in Vietnamese voice samples. Use the CLI to list them.

In [ ]:
!viettts show-voices

## Step 3: Quick Synthesis via CLI

The fastest way to generate speech is with the `viettts synthesis` command:
- `--text`: Vietnamese text to speak
- `--voice`: voice ID (number) or voice name from the built-in list
- `--speed`: speed multiplier (0.5 to 2.0, default 1.0)
- `--output`: output WAV file path

In [ ]:
!viettts synthesis \
    --text "Xin chao! Toi la tro ly giong noi tieng Viet." \
    --voice 1 \
    --speed 1.0 \
    --output cli_output.wav

In [ ]:
from IPython.display import Audio, display

display(Audio("cli_output.wav"))

## Step 4: Python API — Programmatic Usage

For more control, use the `TTS` class directly in Python.

| Method | Description |
|--------|-------------|
| `TTS(model_dir)` | Initialize with model directory (auto-downloads if needed) |
| `list_avaliable_spks()` | List available speaker IDs |
| `tts_to_wav(text, prompt_speech_16k, speed)` | Generate audio as NumPy array |
| `tts_to_file(text, prompt_speech_16k, speed, output_path)` | Generate and save to WAV file |

The `prompt_speech_16k` parameter is a 16kHz audio waveform used as the voice reference for cloning.

In [ ]:
import time
import torch
import torchaudio
import numpy as np
from IPython.display import Audio, display

from viettts.tts import TTS

# Initialize the TTS model (downloads pretrained weights on first run)
print("Loading VietTTS model...")
t0 = time.time()
tts = TTS(model_dir="pretrained-models")
print(f"Model loaded in {time.time() - t0:.1f}s")

# List available speakers
speakers = tts.list_avaliable_spks()
print(f"\nAvailable speakers: {speakers}")

### Helper: Load Reference Audio

VietTTS expects reference audio as a 16kHz mono waveform (NumPy array). This helper loads and resamples any audio file.

In [ ]:
import os
import glob


def load_prompt_speech(audio_path, target_sr=16000):
    """Load an audio file and resample to 16kHz for use as voice prompt."""
    waveform, sr = torchaudio.load(audio_path)
    if sr != target_sr:
        waveform = torchaudio.transforms.Resample(sr, target_sr)(waveform)
    # Convert to mono if stereo
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    return waveform.squeeze(0).numpy()


# Find built-in voice samples
sample_dirs = ["samples", "pretrained-models/samples"]
voice_files = []
for d in sample_dirs:
    if os.path.exists(d):
        voice_files = sorted(glob.glob(os.path.join(d, "*.wav")))
        break

if voice_files:
    print("Built-in voice samples:")
    for f in voice_files:
        print(f"  {os.path.basename(f)}")
else:
    print("No sample directory found. Will use CLI output as reference.")

## Step 5: Generate Vietnamese Speech

Let's synthesize several Vietnamese sentences using the Python API.

In [ ]:
# Select a reference voice
ref_audio_path = voice_files[0] if voice_files else "cli_output.wav"
print(f"Reference voice: {os.path.basename(ref_audio_path)}")
prompt_speech = load_prompt_speech(ref_audio_path)

# Vietnamese text examples
texts = [
    "Xin chao! Day la he thong chuyen doi van ban thanh giong noi tieng Viet.",
    "Viet Nam la mot quoc gia nam o phia dong ban dao Dong Duong.",
    "Tri tue nhan tao dang thay doi cach chung ta song va lam viec moi ngay.",
]

for i, text in enumerate(texts):
    print(f"\n{'=' * 60}")
    print(f"Text {i + 1}: {text}")

    t0 = time.time()
    wav = tts.tts_to_wav(text=text, prompt_speech_16k=prompt_speech, speed=1.0)
    elapsed = time.time() - t0

    duration = len(wav) / 24000
    print(f"Generated in {elapsed:.1f}s | Audio duration: {duration:.1f}s")
    display(Audio(wav, rate=24000))

## Step 6: Compare Different Voices

VietTTS includes multiple built-in Vietnamese voice samples. Let's compare how the same text sounds with different speakers.

In [ ]:
compare_text = "Thanh pho Ho Chi Minh la thanh pho lon nhat va la trung tam kinh te cua Viet Nam."

for voice_path in voice_files[:4]:
    voice_name = os.path.splitext(os.path.basename(voice_path))[0]
    print(f"\n{'=' * 60}")
    print(f"Voice: {voice_name}")

    prompt = load_prompt_speech(voice_path)
    t0 = time.time()
    wav = tts.tts_to_wav(text=compare_text, prompt_speech_16k=prompt, speed=1.0)
    print(f"Generated in {time.time() - t0:.1f}s")
    display(Audio(wav, rate=24000))

## Step 7: Speed Control

VietTTS supports speed adjustment from 0.5x (slow) to 2.0x (fast). Useful for audiobooks (slower) or notifications (faster).

In [ ]:
speed_text = "Hom nay thoi tiet rat dep, nhiet do ngoai troi khoang hai muoi lam do."
prompt = load_prompt_speech(ref_audio_path)

for speed in [0.8, 1.0, 1.3]:
    print(f"\nSpeed: {speed}x")
    t0 = time.time()
    wav = tts.tts_to_wav(text=speed_text, prompt_speech_16k=prompt, speed=speed)
    duration = len(wav) / 24000
    print(f"  Generated in {time.time() - t0:.1f}s | Duration: {duration:.1f}s")
    display(Audio(wav, rate=24000))

## Step 8: Voice Cloning

VietTTS can clone any voice from a short reference audio clip. Provide a WAV/MP3/MP4 file containing 3-10 seconds of clear speech.

**Via CLI:**
```bash
viettts synthesis --text "Xin chao" --voice /path/to/your_voice.wav --output cloned.wav
```

**Via Python API:**

In [ ]:
# Voice cloning example
# To clone your own voice:
# 1. Record 3-10 seconds of clear speech (Vietnamese preferred)
# 2. Save as WAV (any sample rate, will be resampled to 16kHz)
# 3. Upload to Kaggle as a dataset
# 4. Update the path below and uncomment

# my_voice_path = "/kaggle/input/my-voice/recording.wav"
# my_prompt = load_prompt_speech(my_voice_path)
#
# clone_text = "Day la giong noi cua toi duoc nhan ban boi tri tue nhan tao."
# wav = tts.tts_to_wav(text=clone_text, prompt_speech_16k=my_prompt, speed=1.0)
# display(Audio(wav, rate=24000))
#
# # Or save directly to file
# tts.tts_to_file(clone_text, my_prompt, 1.0, "cloned_output.wav")

print("Voice cloning instructions:")
print("1. Record 3-10s of clear speech (Vietnamese preferred)")
print("2. Upload as Kaggle dataset or place in working directory")
print("3. Uncomment the code above and update the path")
print("4. The model will synthesize new text in your cloned voice")

## Step 9: Batch Generation — Save Multiple Files

Use `tts_to_file()` for batch processing. This is convenient for generating audio datasets or pre-rendering content.

In [ ]:
batch_texts = [
    ("greeting", "Xin chao, rat vui duoc gap ban!"),
    ("weather", "Hom nay troi nang dep, nhiet do khoang ba muoi do."),
    ("news", "Kinh te Viet Nam tang truong manh me trong nam vua qua."),
    ("culture", "Ao dai la trang phuc truyen thong cua nguoi Viet Nam."),
    ("tech", "Cong nghe thong tin la nganh phat trien nhanh nhat hien nay."),
]

prompt = load_prompt_speech(ref_audio_path)

for name, text in batch_texts:
    output_path = f"output_{name}.wav"
    t0 = time.time()
    tts.tts_to_file(
        text=text,
        prompt_speech_16k=prompt,
        speed=1.0,
        output_path=output_path,
    )
    print(f"Saved {output_path} ({time.time() - t0:.1f}s) | {text}")
    display(Audio(output_path))

## Step 10: OpenAI-Compatible API Server (Reference)

VietTTS can run as an OpenAI-compatible API server, making it a drop-in replacement for `openai.audio.speech`.

### Start the server
```bash
viettts server --host 0.0.0.0 --port 8298
```

### Use with OpenAI Python client
```python
import os
os.environ["OPENAI_BASE_URL"] = "http://localhost:8298"
os.environ["OPENAI_API_KEY"] = "viet-tts"

from openai import OpenAI
client = OpenAI()

with client.audio.speech.with_streaming_response.create(
    model="tts-1",
    voice="cdteam",
    input="Xin chao Viet Nam.",
    speed=1.0,
    response_format="wav"
) as response:
    response.stream_to_file("openai_output.wav")
```

### Use with cURL
```bash
curl http://localhost:8298/v1/audio/speech \
  -H "Authorization: Bearer viet-tts" \
  -H "Content-Type: application/json" \
  -d '{"model": "tts-1", "input": "Xin chao Viet Nam.", "voice": "1"}' \
  --output speech.wav
```

### Voice cloning via API
```bash
curl http://localhost:8298/v1/tts \
  --form 'text="Xin chao"' \
  --form 'audio_file=@"/path/to/voice.wav"' \
  --output cloned.wav
```

This makes it easy to swap between OpenAI TTS and VietTTS without changing application code.

## Key Takeaways

### VietTTS at a Glance

| Feature | Details |
|---------|--------|
| **Install** | `pip install viet-tts` |
| **Framework** | Built on F5-TTS with HiFi-GAN vocoder |
| **Voice Cloning** | Yes, from 3-10s reference audio |
| **OpenAI API** | Compatible server at `/v1/audio/speech` |
| **CLI** | `viettts synthesis`, `viettts server`, `viettts show-voices` |
| **Python API** | `TTS` class with `tts_to_wav()` and `tts_to_file()` |
| **Speed Control** | 0.5x to 2.0x |
| **GPU** | Required for reasonable inference speed |
| **License** | Apache 2.0 (code) / CC BY-NC (models) |

### When to Use VietTTS
- You need high-quality Vietnamese TTS with voice cloning
- You want an OpenAI-compatible API for existing applications
- You need a simple `pip install` workflow with auto-downloaded models
- You are building Vietnamese voice applications on the F5-TTS framework

### Comparison with Other Vietnamese TTS Options

| Model | Voice Cloning | Install | API Style |
|-------|---------------|---------|----------|
| **VietTTS** | Yes (3-10s) | `pip install viet-tts` | OpenAI-compatible |
| **viXTTS** | Yes (6-30s) | git clone + HF download | Custom |
| **F5-TTS Vietnamese** | Yes (3-10s) | `pip install f5-tts` + HF | Custom |
| **VieNeu-TTS** | Yes (3-5s) | `pip install vieneu` | Custom |
| **Edge TTS** | No (4 preset voices) | `pip install edge-tts` | Custom |
| **Facebook MMS** | No (single speaker) | `pip install transformers` | HuggingFace |